# BRFSS 2015 - 12-model benchmark with SHAP, LIME and attention (FT-Transformer)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Laiba-Gul/explainable-diabetes-prediction/blob/main/notebooks/05_brfss_xai_benchmark.ipynb)

**Status:** supporting research notebook (original Colab work, cleaned). Outputs were cleared to keep the repository small and free of personal metadata; re-run it or see the reproducible results in [`results/`](../results/).

**What it does:** Leakage-safe 70/15/15 protocol, 7-method train-only feature consensus (15 features), SMOTE-NC on training only, classical ML + boosting + MLP + 1D-CNN + TabTransformer + FT-Transformer + TabNet, frozen test gate, SHAP/LIME/attention explanations. Classical/boosting results are reproduced exactly by `configs/brfss_xai.yaml`.

**Data:** run `diabetes-xai download` (or upload the CSV when the notebook asks). The CSVs are checksum-verified and are not stored in Git.


# Early Diabetes Prediction with Leakage-Safe Explainable AI

**Dataset:** CDC BRFSS 2015 Diabetes Health Indicators  
**Models:** Logistic Regression, SVM, Decision Tree, Random Forest, XGBoost, LightGBM, CatBoost, MLP, 1D CNN, TabTransformer, FT-Transformer, and TabNet  
**Proposed explainable model:** FT-Transformer with SHAP, LIME, and attention-weight heatmaps

This notebook implements a strict 70% training / 15% validation / 15% testing protocol. All learned preprocessing, feature selection, scaling, SMOTE-NC, calibration, model fitting, and early stopping occur without access to the test labels. The test set is unlocked once, near the end, after the model registry is frozen.

> **Research integrity note:** Test metrics are not learning curves. Training and validation are monitored across epochs; final test accuracy and log loss are added only as one-time reference lines after training is complete.

## 0. Runtime setup

For final experiments, use **Runtime > Change runtime type > GPU**. The installation cell uses current Colab-compatible package releases. If Colab asks for a runtime restart, restart and resume from the imports cell.

In [ ]:
%pip -q install imbalanced-learn xgboost lightgbm catboost pytorch-tabnet shap lime

In [ ]:
import os, sys, json, math, time, random, hashlib, shutil, warnings, platform
from pathlib import Path
from copy import deepcopy
from collections import OrderedDict

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import mutual_info_classif, RFE
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    matthews_corrcoef, confusion_matrix, log_loss, roc_curve,
    mutual_info_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from imblearn.over_sampling import SMOTENC, SMOTE
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from pytorch_tabnet.tab_model import TabNetClassifier
import shap
from lime.lime_tabular import LimeTabularExplainer

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid", context="notebook")

RANDOM_STATE = 42
TARGET = "Diabetes_binary"
DATA_PATH = Path("/content/diabetes_binary_health_indicators_BRFSS2015.csv")
OUTPUT_DIR = Path("/content/diabetes_xai_outputs")
FIG_DIR = OUTPUT_DIR / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Final research run: leave FAST_MODE=False.
# FAST_MODE=True is only for debugging the pipeline and must not be reported.
FAST_MODE = False
FAST_TRAIN_ROWS = 40_000
N_SELECTED = 15
SAMPLING_STRATEGY = 1.0
THRESHOLD = 0.50
FULL_EPOCHS = 20
FAST_EPOCHS = 3
BATCH_SIZE = 1024
EPOCHS = FAST_EPOCHS if FAST_MODE else FULL_EPOCHS

DROP_EXACT_DUPLICATES = False  # No respondent ID exists; repeated profiles are not proven duplicate people.

random.seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)
tf.keras.utils.set_random_seed(RANDOM_STATE)
torch.manual_seed(RANDOM_STATE)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_STATE)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SOFT_COLORS = ["#AFC8E8", "#F3B58C"]
print("Python:", sys.version.split()[0])
print("TensorFlow:", tf.__version__)
print("PyTorch:", torch.__version__, "| Device:", DEVICE)
print("FAST_MODE:", FAST_MODE)

## 1. Data collection, upload, and integrity audit

In [ ]:
if not DATA_PATH.exists():
    from google.colab import files
    print(f"Upload: {DATA_PATH.name}")
    uploaded = files.upload()
    if DATA_PATH.name not in uploaded:
        raise FileNotFoundError(f"Expected {DATA_PATH.name}; uploaded: {list(uploaded)}")

raw_bytes = DATA_PATH.read_bytes()
sha256 = hashlib.sha256(raw_bytes).hexdigest()
df_raw = pd.read_csv(DATA_PATH)

print("SHA-256:", sha256)
print("Raw shape:", df_raw.shape)
display(df_raw.head())

In [ ]:
EXPECTED_COLUMNS = [
    "Diabetes_binary", "HighBP", "HighChol", "CholCheck", "BMI", "Smoker",
    "Stroke", "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "GenHlth",
    "MentHlth", "PhysHlth", "DiffWalk", "Sex", "Age", "Education", "Income"
]

BINARY_FEATURES = [
    "HighBP", "HighChol", "CholCheck", "Smoker", "Stroke",
    "HeartDiseaseorAttack", "PhysActivity", "Fruits", "Veggies",
    "HvyAlcoholConsump", "AnyHealthcare", "NoDocbcCost", "DiffWalk", "Sex"
]
ORDINAL_FEATURES = ["GenHlth", "Age", "Education", "Income"]
CONTINUOUS_FEATURES = ["BMI", "MentHlth", "PhysHlth"]
CATEGORICAL_FEATURES = BINARY_FEATURES + ORDINAL_FEATURES
ALL_FEATURES = [c for c in EXPECTED_COLUMNS if c != TARGET]
SCALE_FEATURES = CONTINUOUS_FEATURES + ORDINAL_FEATURES

assert df_raw.columns.tolist() == EXPECTED_COLUMNS, "Unexpected schema or column order."
assert set(df_raw[TARGET].dropna().unique()).issubset({0, 1}), "Target must be binary."
assert np.isfinite(df_raw.to_numpy(dtype=float)).all(), "Infinite values detected."

range_checks = {
    **{c: df_raw[c].isin([0, 1]).all() for c in BINARY_FEATURES + [TARGET]},
    "GenHlth": df_raw["GenHlth"].between(1, 5).all(),
    "Age": df_raw["Age"].between(1, 13).all(),
    "Education": df_raw["Education"].between(1, 6).all(),
    "Income": df_raw["Income"].between(1, 8).all(),
    "MentHlth": df_raw["MentHlth"].between(0, 30).all(),
    "PhysHlth": df_raw["PhysHlth"].between(0, 30).all(),
    "BMI": df_raw["BMI"].gt(0).all(),
}
assert all(range_checks.values()), {k: v for k, v in range_checks.items() if not v}

audit = pd.Series({
    "rows": len(df_raw),
    "columns": df_raw.shape[1],
    "missing_cells": int(df_raw.isna().sum().sum()),
    "exact_repeated_full_rows": int(df_raw.duplicated().sum()),
    "class_0": int((df_raw[TARGET] == 0).sum()),
    "class_1": int((df_raw[TARGET] == 1).sum()),
})
display(audit.to_frame("value"))

df = df_raw.drop_duplicates().copy() if DROP_EXACT_DUPLICATES else df_raw.copy()
df[TARGET] = df[TARGET].astype("int8")
for c in BINARY_FEATURES:
    df[c] = df[c].astype("int8")
for c in ORDINAL_FEATURES + ["MentHlth", "PhysHlth"]:
    df[c] = df[c].astype("int16")
df["BMI"] = df["BMI"].astype("float32")
print("Analysis shape:", df.shape, "| DROP_EXACT_DUPLICATES:", DROP_EXACT_DUPLICATES)

In [ ]:
counts = df[TARGET].value_counts().sort_index()
fig, ax = plt.subplots(figsize=(7, 4.5))
ax.pie(
    counts.values,
    labels=["No diabetes (0)", "Prediabetes/diabetes (1)"],
    colors=SOFT_COLORS,
    startangle=90,
    counterclock=False,
    autopct=lambda p: f"{p:.2f}%\n({int(round(p * counts.sum() / 100)):,})",
    wedgeprops={"edgecolor": "white", "linewidth": 1.5},
)
ax.set_title("BRFSS 2015 Diabetes Class Distribution", weight="bold")
ax.axis("equal")
plt.tight_layout()
plt.savefig(FIG_DIR / "01_overall_class_distribution_pie.png", dpi=220, bbox_inches="tight")
plt.show()

## 2. Stratified 70% / 15% / 15% split

The split occurs before any learned transform. `train_test_split` is called twice with stratification. Index assertions prove that the partitions are disjoint.

In [ ]:
X = df[ALL_FEATURES].copy()
y = df[TARGET].astype(int).copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

assert set(X_train.index).isdisjoint(X_val.index)
assert set(X_train.index).isdisjoint(X_test.index)
assert set(X_val.index).isdisjoint(X_test.index)
assert len(X_train) + len(X_val) + len(X_test) == len(df)

split_summary = []
for name, labels in [("Training", y_train), ("Validation", y_val)]:
    vc = labels.value_counts().sort_index()
    split_summary.append({
        "Split": name, "Rows": len(labels), "Percent": len(labels) / len(df),
        "Class 0": int(vc.get(0, 0)), "Class 1": int(vc.get(1, 0)),
        "Class 1 rate": float(labels.mean())
    })
split_summary.append({
    "Split": "Testing (locked)", "Rows": len(X_test), "Percent": len(X_test) / len(df),
    "Class 0": "Locked", "Class 1": "Locked", "Class 1 rate": np.nan
})
split_summary = pd.DataFrame(split_summary)
display(split_summary.style.format({"Percent": "{:.1%}", "Class 1 rate": "{:.2%}"}))

# y_test is created to form the stratified split, then remains unused until the FINAL TEST GATE.
TEST_RESULTS_UNLOCKED = False

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, (name, labels) in zip(axes, [("Training", y_train), ("Validation", y_val)]):
    vc = labels.value_counts().sort_index()
    ax.pie(vc.values, labels=["0", "1"], colors=SOFT_COLORS, startangle=90,
           counterclock=False, autopct="%1.1f%%", wedgeprops={"edgecolor": "white"})
    ax.set_title(f"{name} class distribution")
    ax.axis("equal")
plt.tight_layout()
plt.savefig(FIG_DIR / "02_split_class_distribution_pies.png", dpi=220, bbox_inches="tight")
plt.show()

## 3. Train-only feature selection analysis

Seven methods are computed on the original, unbalanced training partition: correlation, Random Forest feature importance, Information Gain, Gain Ratio, Mutual Information, RFE, and TreeSHAP. Consensus ranking is the mean of the seven method ranks. No validation or test values are used to fit these methods.

In [ ]:
def maybe_fast_training_subset(X_frame, y_series, n=FAST_TRAIN_ROWS):
    if not FAST_MODE or len(X_frame) <= n:
        return X_frame.copy(), y_series.copy()
    X_sub, _, y_sub, _ = train_test_split(
        X_frame, y_series, train_size=n, stratify=y_series, random_state=RANDOM_STATE
    )
    return X_sub.copy(), y_sub.copy()

X_fs, y_fs = maybe_fast_training_subset(X_train, y_train)

fs_scaler = StandardScaler().fit(X_train[SCALE_FEATURES])
X_train_scaled_for_fs = X_fs.copy()
X_train_scaled_for_fs[SCALE_FEATURES] = fs_scaler.transform(X_fs[SCALE_FEATURES])

# 1) Pearson / point-biserial correlation.
corr_scores = X_fs.apply(lambda s: s.corr(y_fs)).fillna(0.0)

# Correlation heatmap fitted on training data only.
train_corr = pd.concat([X_fs, y_fs.rename(TARGET)], axis=1).corr(numeric_only=True)
plt.figure(figsize=(14, 11))
sns.heatmap(train_corr, cmap="vlag", center=0, square=True, cbar_kws={"shrink": 0.7})
plt.title("Training-Only Correlation Matrix")
plt.tight_layout()
plt.savefig(FIG_DIR / "03_training_correlation_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

# 2) Random Forest Gini importance.
rf_fs = RandomForestClassifier(
    n_estimators=120 if FAST_MODE else 300,
    class_weight="balanced", min_samples_leaf=3,
    random_state=RANDOM_STATE, n_jobs=-1
)
rf_fs.fit(X_train_scaled_for_fs, y_fs)
rf_scores = pd.Series(rf_fs.feature_importances_, index=ALL_FEATURES)

# 3) Information Gain and 4) Gain Ratio.
def train_discrete_codes(series, continuous):
    if continuous:
        return pd.qcut(series, q=10, labels=False, duplicates="drop").fillna(-1).astype(int)
    return series.astype(int)

ig_scores, gr_scores = {}, {}
for feature in ALL_FEATURES:
    codes = train_discrete_codes(X_fs[feature], feature in CONTINUOUS_FEATURES)
    ig = mutual_info_score(codes, y_fs)
    intrinsic = mutual_info_score(codes, codes)
    ig_scores[feature] = ig
    gr_scores[feature] = ig / intrinsic if intrinsic > 0 else 0.0
ig_scores = pd.Series(ig_scores)
gr_scores = pd.Series(gr_scores)

# 5) Mutual Information with a mixed discrete/continuous mask.
discrete_mask = np.array([c in CATEGORICAL_FEATURES for c in ALL_FEATURES])
mi = mutual_info_classif(
    X_train_scaled_for_fs[ALL_FEATURES], y_fs,
    discrete_features=discrete_mask, random_state=RANDOM_STATE
)
mi_scores = pd.Series(mi, index=ALL_FEATURES)

# 6) Recursive Feature Elimination with balanced Logistic Regression.
rfe_estimator = LogisticRegression(
    class_weight="balanced", solver="liblinear", max_iter=2000, random_state=RANDOM_STATE
)
rfe = RFE(rfe_estimator, n_features_to_select=N_SELECTED, step=1)
rfe.fit(X_train_scaled_for_fs[ALL_FEATURES], y_fs)
rfe_rank = pd.Series(rfe.ranking_, index=ALL_FEATURES)

print("Completed correlation, RF importance, Information Gain, Gain Ratio, MI, and RFE.")

In [ ]:
# 7) Train-only XGBoost + TreeSHAP feature importance.
negative, positive = np.bincount(y_fs)
shap_fs_model = XGBClassifier(
    n_estimators=180 if FAST_MODE else 350,
    max_depth=4, learning_rate=0.05,
    subsample=0.85, colsample_bytree=0.85,
    objective="binary:logistic", eval_metric="logloss",
    tree_method="hist", scale_pos_weight=negative / positive,
    random_state=RANDOM_STATE, n_jobs=-1
)
shap_fs_model.fit(X_train_scaled_for_fs[ALL_FEATURES], y_fs)

shap_sample = X_train_scaled_for_fs[ALL_FEATURES].sample(
    n=min(1500 if not FAST_MODE else 400, len(X_train_scaled_for_fs)),
    random_state=RANDOM_STATE
)
tree_explainer = shap.TreeExplainer(shap_fs_model)
tree_sv = tree_explainer.shap_values(shap_sample)
if isinstance(tree_sv, list):
    tree_sv = tree_sv[-1]
tree_sv = np.asarray(tree_sv)
if tree_sv.ndim == 3:
    tree_sv = tree_sv[:, :, -1]
shap_scores = pd.Series(np.abs(tree_sv).mean(axis=0), index=ALL_FEATURES)

feature_results = pd.DataFrame({
    "Feature": ALL_FEATURES,
    "Pearson r": corr_scores.reindex(ALL_FEATURES).values,
    "RF Importance": rf_scores.reindex(ALL_FEATURES).values,
    "Information Gain": ig_scores.reindex(ALL_FEATURES).values,
    "Gain Ratio": gr_scores.reindex(ALL_FEATURES).values,
    "Mutual Information": mi_scores.reindex(ALL_FEATURES).values,
    "RFE Rank": rfe_rank.reindex(ALL_FEATURES).values,
    "SHAP Importance": shap_scores.reindex(ALL_FEATURES).values,
})

rank_specs = {
    "|Pearson| Rank": feature_results["Pearson r"].abs().rank(ascending=False, method="average"),
    "RF Rank": feature_results["RF Importance"].rank(ascending=False, method="average"),
    "IG Rank": feature_results["Information Gain"].rank(ascending=False, method="average"),
    "Gain Ratio Rank": feature_results["Gain Ratio"].rank(ascending=False, method="average"),
    "MI Rank": feature_results["Mutual Information"].rank(ascending=False, method="average"),
    "RFE Method Rank": feature_results["RFE Rank"].rank(ascending=True, method="average"),
    "SHAP Rank": feature_results["SHAP Importance"].rank(ascending=False, method="average"),
}
for column, values in rank_specs.items():
    feature_results[column] = values
rank_columns = list(rank_specs)
feature_results["Consensus Rank"] = feature_results[rank_columns].mean(axis=1)
feature_results = feature_results.sort_values(["Consensus Rank", "Feature"]).reset_index(drop=True)
SELECTED_FEATURES = feature_results.head(N_SELECTED)["Feature"].tolist()

display(feature_results.round(6))
print("Frozen selected features:", SELECTED_FEATURES)
feature_results.to_csv(OUTPUT_DIR / "feature_selection_results.csv", index=False)
(OUTPUT_DIR / "selected_features.json").write_text(json.dumps({
    "selected_features": SELECTED_FEATURES,
    "n_selected": N_SELECTED,
    "random_state": RANDOM_STATE,
    "fast_mode": FAST_MODE,
}, indent=2))

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 11))
plot_specs = [
    ("Pearson r", "Pearson correlation", "coolwarm"),
    ("RF Importance", "Random Forest importance", "Blues_r"),
    ("Information Gain", "Information Gain", "Greens_r"),
    ("SHAP Importance", "TreeSHAP mean |value|", "Oranges_r"),
]
for ax, (column, title, palette) in zip(axes.ravel(), plot_specs):
    data = feature_results.sort_values(column, ascending=True).tail(15)
    sns.barplot(data=data, x=column, y="Feature", ax=ax, palette=palette, hue="Feature", legend=False)
    ax.set_title(title)
plt.tight_layout()
plt.savefig(FIG_DIR / "04_feature_selection_methods.png", dpi=220, bbox_inches="tight")
plt.show()

plt.figure(figsize=(9, 6))
ranked = feature_results.sort_values("Consensus Rank", ascending=False)
sns.barplot(data=ranked, x="Consensus Rank", y="Feature", color="#6E9BC9")
plt.title("Consensus Feature Rank (Lower Is Better)")
plt.tight_layout()
plt.savefig(FIG_DIR / "05_consensus_feature_rank.png", dpi=220, bbox_inches="tight")
plt.show()

## 4. SMOTE-NC on training only and train-only model scaling

SMOTE-NC is appropriate for mixed continuous and categorical variables. Continuous variables are temporarily standardized using a scaler fitted on original training data. Categorical variables keep valid levels. Validation and test rows are never passed to `fit_resample`.

In [ ]:
selected_categorical = [c for c in SELECTED_FEATURES if c in CATEGORICAL_FEATURES]
selected_continuous = [c for c in SELECTED_FEATURES if c in CONTINUOUS_FEATURES]
selected_scale = [c for c in SELECTED_FEATURES if c in SCALE_FEATURES]

X_train_selected = X_train[SELECTED_FEATURES].copy()
X_val_selected = X_val[SELECTED_FEATURES].copy()
X_test_selected = X_test[SELECTED_FEATURES].copy()  # Features only; labels remain locked.

# Optional FAST_MODE subset is drawn from training only.
X_train_fit, y_train_fit = maybe_fast_training_subset(X_train_selected, y_train)

# Scale continuous dimensions for the SMOTE-NC neighbor space using original training only.
smote_cont_scaler = None
X_smote_space = X_train_fit.copy()
if selected_continuous:
    smote_cont_scaler = StandardScaler().fit(X_train_fit[selected_continuous])
    X_smote_space[selected_continuous] = smote_cont_scaler.transform(X_train_fit[selected_continuous])

categorical_indices = [X_smote_space.columns.get_loc(c) for c in selected_categorical]
if categorical_indices:
    sampler = SMOTENC(
        categorical_features=categorical_indices,
        sampling_strategy=SAMPLING_STRATEGY,
        k_neighbors=5,
        random_state=RANDOM_STATE,
    )
else:
    sampler = SMOTE(
        sampling_strategy=SAMPLING_STRATEGY,
        k_neighbors=5,
        random_state=RANDOM_STATE,
    )

X_train_bal_space, y_train_bal = sampler.fit_resample(X_smote_space, y_train_fit)
X_train_bal_raw = pd.DataFrame(X_train_bal_space, columns=SELECTED_FEATURES)
if selected_continuous:
    X_train_bal_raw[selected_continuous] = smote_cont_scaler.inverse_transform(
        X_train_bal_raw[selected_continuous]
    )
for c in selected_categorical:
    X_train_bal_raw[c] = X_train_bal_raw[c].round().astype(int)

# Assert valid categorical levels against original training categories.
for c in selected_categorical:
    assert set(X_train_bal_raw[c].unique()).issubset(set(X_train_fit[c].astype(int).unique()))

# Final model scaler is fit on original training rows, never synthetic/validation/test rows.
model_scaler = StandardScaler().fit(X_train_fit[selected_scale]) if selected_scale else None

def transform_for_models(frame):
    out = frame[SELECTED_FEATURES].copy()
    if selected_scale:
        out[selected_scale] = model_scaler.transform(out[selected_scale])
    return out.astype("float32")

X_train_bal_model = transform_for_models(X_train_bal_raw)
X_train_eval_model = transform_for_models(X_train_selected)
X_val_model = transform_for_models(X_val_selected)
X_test_model = transform_for_models(X_test_selected)
y_train_bal = pd.Series(y_train_bal).astype(int).reset_index(drop=True)

print("Training before:", y_train_fit.value_counts().sort_index().to_dict())
print("Training after SMOTE-NC:", y_train_bal.value_counts().sort_index().to_dict())
print("Validation untouched:", y_val.value_counts().sort_index().to_dict())
print("Test held out:", len(X_test), "rows (labels locked until final evaluation)")

In [ ]:
before = y_train_fit.value_counts().sort_index()
after = y_train_bal.value_counts().sort_index()
fig, axes = plt.subplots(1, 2, figsize=(9, 4))
for ax, series, title in [
    (axes[0], before, "Training before SMOTE-NC"),
    (axes[1], after, "Training after SMOTE-NC"),
]:
    ax.pie(series.values, labels=["Class 0", "Class 1"], colors=SOFT_COLORS,
           startangle=90, counterclock=False, autopct="%1.1f%%",
           wedgeprops={"edgecolor": "white"})
    ax.set_title(title)
    ax.axis("equal")
plt.tight_layout()
plt.savefig(FIG_DIR / "06_training_before_after_smote_pies.png", dpi=220, bbox_inches="tight")
plt.show()

## 5. Common evaluation functions - only the requested metrics

In [ ]:
METRIC_COLUMNS = [
    "Accuracy", "Precision", "Recall", "F1-score", "ROC-AUC",
    "MCC", "Specificity", "Sensitivity", "Log Loss"
]

def requested_metrics(y_true, probabilities, threshold=THRESHOLD):
    y_true = np.asarray(y_true).astype(int)
    probabilities = np.clip(np.asarray(probabilities).reshape(-1), 1e-7, 1 - 1e-7)
    predictions = (probabilities >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, predictions, labels=[0, 1]).ravel()
    specificity = tn / (tn + fp) if (tn + fp) else 0.0
    sensitivity = tp / (tp + fn) if (tp + fn) else 0.0
    return OrderedDict([
        ("Accuracy", accuracy_score(y_true, predictions)),
        ("Precision", precision_score(y_true, predictions, zero_division=0)),
        ("Recall", recall_score(y_true, predictions, zero_division=0)),
        ("F1-score", f1_score(y_true, predictions, zero_division=0)),
        ("ROC-AUC", roc_auc_score(y_true, probabilities)),
        ("MCC", matthews_corrcoef(y_true, predictions)),
        ("Specificity", specificity),
        ("Sensitivity", sensitivity),
        ("Log Loss", log_loss(y_true, probabilities, labels=[0, 1])),
    ])

def positive_probability(model, X_matrix):
    return np.asarray(model.predict_proba(X_matrix))[:, 1]

train_val_rows = []
validation_probabilities = {}
model_registry = OrderedDict()
model_families = {}

def record_train_validation(name, family, train_prob, val_prob):
    validation_probabilities[name] = np.asarray(val_prob).reshape(-1)
    for split_name, labels, probs in [
        ("Training", y_train, train_prob),
        ("Validation", y_val, val_prob),
    ]:
        row = {"Model": name, "Family": family, "Split": split_name}
        row.update(requested_metrics(labels, probs))
        train_val_rows.append(row)

## 6. Classical machine-learning and ensemble models

In [ ]:
classical_models = OrderedDict({
    "Logistic Regression": LogisticRegression(max_iter=2000, solver="lbfgs", random_state=RANDOM_STATE),
    "SVM (calibrated LinearSVC)": CalibratedClassifierCV(
        estimator=LinearSVC(C=1.0, random_state=RANDOM_STATE),
        method="sigmoid", cv=3, n_jobs=-1
    ),
    "Decision Tree": DecisionTreeClassifier(
        max_depth=10, min_samples_leaf=20, random_state=RANDOM_STATE
    ),
})

ensemble_models = OrderedDict({
    "Random Forest": RandomForestClassifier(
        n_estimators=180 if FAST_MODE else 400, min_samples_leaf=2,
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": XGBClassifier(
        n_estimators=180 if FAST_MODE else 500, max_depth=5, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85, objective="binary:logistic",
        eval_metric="logloss", tree_method="hist", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "LightGBM": LGBMClassifier(
        n_estimators=180 if FAST_MODE else 500, learning_rate=0.05,
        num_leaves=31, subsample=0.85, colsample_bytree=0.85,
        random_state=RANDOM_STATE, n_jobs=-1, verbosity=-1
    ),
    "CatBoost": CatBoostClassifier(
        iterations=180 if FAST_MODE else 500, depth=7, learning_rate=0.05,
        loss_function="Logloss", random_seed=RANDOM_STATE,
        verbose=False, allow_writing_files=False
    ),
})

for family, candidates in [("Machine Learning", classical_models), ("Ensemble", ensemble_models)]:
    for name, model in candidates.items():
        started = time.time()
        model.fit(X_train_bal_model, y_train_bal)
        model_registry[name] = model
        model_families[name] = family
        train_prob = positive_probability(model, X_train_eval_model)
        val_prob = positive_probability(model, X_val_model)
        record_train_validation(name, family, train_prob, val_prob)
        print(f"{name}: {time.time() - started:.1f}s")

train_val_results = pd.DataFrame(train_val_rows)
display(train_val_results[train_val_results["Split"] == "Validation"].sort_values("ROC-AUC", ascending=False).round(4))

In [ ]:
def plot_confusion_grid(probability_map, labels, title, filename):
    n = len(probability_map)
    cols = 3
    rows = math.ceil(n / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(14, 4.2 * rows))
    axes = np.atleast_1d(axes).ravel()
    for ax, (name, probs) in zip(axes, probability_map.items()):
        preds = (np.asarray(probs) >= THRESHOLD).astype(int)
        cm = confusion_matrix(labels, preds, labels=[0, 1])
        sns.heatmap(cm, annot=True, fmt=",d", cmap="Blues", cbar=False, ax=ax)
        ax.set_title(name)
        ax.set_xlabel("Predicted")
        ax.set_ylabel("Actual")
    for ax in axes[n:]:
        ax.axis("off")
    fig.suptitle(title, y=1.01, fontsize=14, weight="bold")
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

plot_confusion_grid(
    {k: validation_probabilities[k] for k in list(classical_models) + list(ensemble_models)},
    y_val, "Validation Confusion Matrices - ML and Ensemble Models",
    "07_validation_confusion_ml_ensemble.png"
)

## 7. Deep learning: MLP and 1D CNN

In [ ]:
X_bal_np = X_train_bal_model.to_numpy(dtype=np.float32)
y_bal_np = y_train_bal.to_numpy(dtype=np.float32)
X_train_eval_np = X_train_eval_model.to_numpy(dtype=np.float32)
X_val_np = X_val_model.to_numpy(dtype=np.float32)
X_test_np = X_test_model.to_numpy(dtype=np.float32)

def build_mlp(n_features):
    inputs = keras.Input(shape=(n_features,))
    x = layers.Dense(128, activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(64, activation="relu")(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.20)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="MLP")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return model

def build_1d_cnn(n_features):
    inputs = keras.Input(shape=(n_features, 1))
    x = layers.Conv1D(64, 3, padding="same", activation="relu")(inputs)
    x = layers.BatchNormalization()(x)
    x = layers.MaxPooling1D(2)(x)
    x = layers.Conv1D(32, 3, padding="same", activation="relu")(x)
    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dropout(0.30)(x)
    x = layers.Dense(32, activation="relu")(x)
    outputs = layers.Dense(1, activation="sigmoid")(x)
    model = keras.Model(inputs, outputs, name="CNN1D")
    model.compile(optimizer=keras.optimizers.Adam(1e-3), loss="binary_crossentropy", metrics=["accuracy"])
    return model

keras_models = OrderedDict()
keras_histories = OrderedDict()

def fit_keras_candidate(name, model, train_x, val_x, train_eval_x):
    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
        keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
    ]
    history = model.fit(
        train_x, y_bal_np,
        validation_data=(val_x, y_val.to_numpy()),
        epochs=EPOCHS, batch_size=BATCH_SIZE,
        callbacks=callbacks, verbose=1
    )
    keras_models[name] = model
    keras_histories[name] = history.history
    model_registry[name] = model
    model_families[name] = "Deep Learning"
    train_prob = model.predict(train_eval_x, batch_size=4096, verbose=0).reshape(-1)
    val_prob = model.predict(val_x, batch_size=4096, verbose=0).reshape(-1)
    record_train_validation(name, "Deep Learning", train_prob, val_prob)

mlp = build_mlp(len(SELECTED_FEATURES))
fit_keras_candidate("MLP", mlp, X_bal_np, X_val_np, X_train_eval_np)

cnn = build_1d_cnn(len(SELECTED_FEATURES))
fit_keras_candidate(
    "1D CNN", cnn,
    X_bal_np[..., None], X_val_np[..., None], X_train_eval_np[..., None]
)

In [ ]:
def plot_keras_history(histories, filename):
    fig, axes = plt.subplots(len(histories), 2, figsize=(12, 4 * len(histories)))
    axes = np.atleast_2d(axes)
    for row, (name, hist) in enumerate(histories.items()):
        axes[row, 0].plot(hist["accuracy"], label="Training")
        axes[row, 0].plot(hist["val_accuracy"], label="Validation")
        axes[row, 0].set_title(f"{name}: Accuracy")
        axes[row, 0].set_xlabel("Epoch")
        axes[row, 0].legend()
        axes[row, 1].plot(hist["loss"], label="Training")
        axes[row, 1].plot(hist["val_loss"], label="Validation")
        axes[row, 1].set_title(f"{name}: Loss")
        axes[row, 1].set_xlabel("Epoch")
        axes[row, 1].legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

plot_keras_history(keras_histories, "08_deep_learning_train_validation_curves.png")

## 8. Transformer data preparation

Binary and ordinal predictors are categorical tokens. Category mappings and the continuous scaler are learned from original training rows only. Unknown categories map to index 0.

In [ ]:
TRANSFORMER_CAT = [c for c in SELECTED_FEATURES if c in CATEGORICAL_FEATURES]
TRANSFORMER_NUM = [c for c in SELECTED_FEATURES if c not in TRANSFORMER_CAT]

category_maps = {}
category_cardinalities = []
for c in TRANSFORMER_CAT:
    levels = sorted(pd.Series(X_train_fit[c]).astype(int).unique().tolist())
    category_maps[c] = {value: idx + 1 for idx, value in enumerate(levels)}  # 0 = unknown
    category_cardinalities.append(len(levels) + 1)

transformer_num_scaler = StandardScaler().fit(X_train_fit[TRANSFORMER_NUM]) if TRANSFORMER_NUM else None

def make_transformer_arrays(frame):
    if TRANSFORMER_CAT:
        x_cat = np.column_stack([
            frame[c].round().astype(int).map(category_maps[c]).fillna(0).astype(int).to_numpy()
            for c in TRANSFORMER_CAT
        ]).astype(np.int64)
    else:
        x_cat = np.zeros((len(frame), 0), dtype=np.int64)
    if TRANSFORMER_NUM:
        x_num = transformer_num_scaler.transform(frame[TRANSFORMER_NUM]).astype(np.float32)
    else:
        x_num = np.zeros((len(frame), 0), dtype=np.float32)
    return x_cat, x_num

X_bal_cat, X_bal_num = make_transformer_arrays(X_train_bal_raw)
X_train_cat, X_train_num = make_transformer_arrays(X_train_selected)
X_val_cat, X_val_num = make_transformer_arrays(X_val_selected)
X_test_cat, X_test_num = make_transformer_arrays(X_test_selected)

def make_loader(x_cat, x_num, labels, shuffle=False, batch_size=BATCH_SIZE):
    ds = TensorDataset(
        torch.as_tensor(x_cat, dtype=torch.long),
        torch.as_tensor(x_num, dtype=torch.float32),
        torch.as_tensor(np.asarray(labels), dtype=torch.float32),
    )
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0,
                      pin_memory=torch.cuda.is_available())

train_bal_loader = make_loader(X_bal_cat, X_bal_num, y_train_bal, shuffle=True)
train_eval_loader = make_loader(X_train_cat, X_train_num, y_train, shuffle=False, batch_size=4096)
val_loader = make_loader(X_val_cat, X_val_num, y_val, shuffle=False, batch_size=4096)
test_feature_loader = make_loader(
    X_test_cat, X_test_num, np.zeros(len(X_test_cat)), shuffle=False, batch_size=4096
)

print("Categorical:", TRANSFORMER_CAT)
print("Numerical:", TRANSFORMER_NUM)

## 9. TabTransformer and FT-Transformer architectures

In [ ]:
class AttentionBlock(nn.Module):
    def __init__(self, d_token=32, n_heads=4, dropout=0.15, ff_mult=2):
        super().__init__()
        self.norm1 = nn.LayerNorm(d_token)
        self.attn = nn.MultiheadAttention(d_token, n_heads, dropout=dropout, batch_first=True)
        self.norm2 = nn.LayerNorm(d_token)
        self.ff = nn.Sequential(
            nn.Linear(d_token, d_token * ff_mult), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(d_token * ff_mult, d_token), nn.Dropout(dropout)
        )

    def forward(self, x, return_attention=False):
        z = self.norm1(x)
        attn_out, attn_weights = self.attn(
            z, z, z, need_weights=return_attention, average_attn_weights=False
        )
        x = x + attn_out
        x = x + self.ff(self.norm2(x))
        return x, attn_weights


class TabTransformer(nn.Module):
    def __init__(self, cardinalities, n_num, d_token=32, depth=2, n_heads=4, dropout=0.15):
        super().__init__()
        self.embeddings = nn.ModuleList([nn.Embedding(card, d_token) for card in cardinalities])
        self.blocks = nn.ModuleList([AttentionBlock(d_token, n_heads, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(d_token)
        input_dim = len(cardinalities) * d_token + n_num
        self.head = nn.Sequential(
            nn.Linear(input_dim, 128), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(128, 64), nn.ReLU(), nn.Dropout(dropout),
            nn.Linear(64, 1)
        )

    def forward(self, x_cat, x_num):
        if self.embeddings:
            tokens = torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
            for block in self.blocks:
                tokens, _ = block(tokens, return_attention=False)
            cat_repr = self.norm(tokens).flatten(1)
        else:
            cat_repr = x_num.new_zeros((x_num.shape[0], 0))
        combined = torch.cat([cat_repr, x_num], dim=1)
        return self.head(combined).squeeze(1)


class FTTransformer(nn.Module):
    def __init__(self, cardinalities, n_num, d_token=32, depth=3, n_heads=4, dropout=0.15):
        super().__init__()
        self.cat_embeddings = nn.ModuleList([nn.Embedding(card, d_token) for card in cardinalities])
        self.n_num = n_num
        if n_num:
            self.num_weight = nn.Parameter(torch.randn(n_num, d_token) * 0.02)
            self.num_bias = nn.Parameter(torch.zeros(n_num, d_token))
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_token))
        nn.init.normal_(self.cls_token, std=0.02)
        self.blocks = nn.ModuleList([AttentionBlock(d_token, n_heads, dropout) for _ in range(depth)])
        self.norm = nn.LayerNorm(d_token)
        self.head = nn.Sequential(nn.Linear(d_token, 64), nn.ReLU(), nn.Dropout(dropout), nn.Linear(64, 1))

    def forward(self, x_cat, x_num, return_attention=False):
        token_parts = []
        if self.cat_embeddings:
            token_parts.append(torch.stack([
                emb(x_cat[:, i]) for i, emb in enumerate(self.cat_embeddings)
            ], dim=1))
        if self.n_num:
            token_parts.append(x_num.unsqueeze(-1) * self.num_weight.unsqueeze(0) + self.num_bias.unsqueeze(0))
        tokens = torch.cat(token_parts, dim=1)
        cls = self.cls_token.expand(tokens.shape[0], -1, -1)
        x = torch.cat([cls, tokens], dim=1)
        attentions = []
        for block in self.blocks:
            x, attn = block(x, return_attention=return_attention)
            if return_attention:
                attentions.append(attn)
        logits = self.head(self.norm(x[:, 0])).squeeze(1)
        return (logits, attentions) if return_attention else logits

In [ ]:
def predict_torch(model, loader):
    model.eval()
    probs = []
    with torch.no_grad():
        for x_cat, x_num, _ in loader:
            logits = model(x_cat.to(DEVICE), x_num.to(DEVICE))
            probs.append(torch.sigmoid(logits).cpu().numpy())
    return np.concatenate(probs)

def fit_torch_candidate(name, model, max_epochs=EPOCHS, patience=4):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-5)
    criterion = nn.BCEWithLogitsLoss()
    best_state, best_val, stale = None, float("inf"), 0
    history = {"accuracy": [], "val_accuracy": [], "loss": [], "val_loss": []}

    for epoch in range(max_epochs):
        model.train()
        train_losses, train_correct, train_total = [], 0, 0
        for x_cat, x_num, labels in train_bal_loader:
            x_cat, x_num, labels = x_cat.to(DEVICE), x_num.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad(set_to_none=True)
            logits = model(x_cat, x_num)
            loss = criterion(logits, labels)
            loss.backward()
            optimizer.step()
            train_losses.append(loss.item() * len(labels))
            train_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
            train_total += len(labels)

        model.eval()
        val_losses, val_correct, val_total = [], 0, 0
        with torch.no_grad():
            for x_cat, x_num, labels in val_loader:
                x_cat, x_num, labels = x_cat.to(DEVICE), x_num.to(DEVICE), labels.to(DEVICE)
                logits = model(x_cat, x_num)
                loss = criterion(logits, labels)
                val_losses.append(loss.item() * len(labels))
                val_correct += ((torch.sigmoid(logits) >= 0.5) == labels.bool()).sum().item()
                val_total += len(labels)

        train_loss = sum(train_losses) / train_total
        val_loss_value = sum(val_losses) / val_total
        history["loss"].append(train_loss)
        history["val_loss"].append(val_loss_value)
        history["accuracy"].append(train_correct / train_total)
        history["val_accuracy"].append(val_correct / val_total)
        print(f"{name} epoch {epoch+1:02d}: loss={train_loss:.4f}, val_loss={val_loss_value:.4f}")

        if val_loss_value < best_val - 1e-5:
            best_val = val_loss_value
            best_state = deepcopy(model.state_dict())
            stale = 0
        else:
            stale += 1
            if stale >= patience:
                print("Early stopping")
                break

    model.load_state_dict(best_state)
    train_prob = predict_torch(model, train_eval_loader)
    val_prob = predict_torch(model, val_loader)
    model_registry[name] = model
    model_families[name] = "Transformer"
    validation_probabilities[name] = val_prob
    record_train_validation(name, "Transformer", train_prob, val_prob)
    return model, history

tab_transformer = TabTransformer(category_cardinalities, len(TRANSFORMER_NUM))
tab_transformer, tab_transformer_history = fit_torch_candidate("TabTransformer", tab_transformer)

ft_transformer = FTTransformer(category_cardinalities, len(TRANSFORMER_NUM))
ft_transformer, ft_transformer_history = fit_torch_candidate("FT-Transformer", ft_transformer)

torch_histories = OrderedDict({
    "TabTransformer": tab_transformer_history,
    "FT-Transformer": ft_transformer_history,
})

## 10. TabNet

In [ ]:
tabnet = TabNetClassifier(
    n_d=24, n_a=24, n_steps=5, gamma=1.5,
    lambda_sparse=1e-4,
    optimizer_fn=torch.optim.Adam,
    optimizer_params=dict(lr=2e-3),
    mask_type="entmax",
    seed=RANDOM_STATE,
    verbose=10,
    device_name="cuda" if torch.cuda.is_available() else "cpu",
)
tabnet.fit(
    X_train=X_bal_np,
    y_train=y_bal_np.astype(int),
    eval_set=[(X_bal_np, y_bal_np.astype(int)), (X_val_np, y_val.to_numpy())],
    eval_name=["train", "val"],
    eval_metric=["logloss", "accuracy"],
    max_epochs=EPOCHS,
    patience=4,
    batch_size=8192 if not FAST_MODE else 2048,
    virtual_batch_size=1024 if not FAST_MODE else 256,
    num_workers=0,
    drop_last=False,
)
model_registry["TabNet"] = tabnet
model_families["TabNet"] = "Transformer"
train_prob = tabnet.predict_proba(X_train_eval_np)[:, 1]
val_prob = tabnet.predict_proba(X_val_np)[:, 1]
record_train_validation("TabNet", "Transformer", train_prob, val_prob)
tabnet_history = dict(tabnet.history)
tabnet_curve_history = {
    "accuracy": tabnet_history.get("train_accuracy", []),
    "val_accuracy": tabnet_history.get("val_accuracy", []),
    "loss": tabnet_history.get("train_logloss", tabnet_history.get("loss", [])),
    "val_loss": tabnet_history.get("val_logloss", []),
}

In [ ]:
def plot_generic_histories(histories, filename):
    fig, axes = plt.subplots(len(histories), 2, figsize=(12, 4 * len(histories)))
    axes = np.atleast_2d(axes)
    for row, (name, hist) in enumerate(histories.items()):
        axes[row, 0].plot(hist["accuracy"], label="Training")
        axes[row, 0].plot(hist["val_accuracy"], label="Validation")
        axes[row, 0].set_title(f"{name}: Accuracy")
        axes[row, 0].set_xlabel("Epoch")
        axes[row, 0].legend()
        axes[row, 1].plot(hist["loss"], label="Training")
        axes[row, 1].plot(hist["val_loss"], label="Validation")
        axes[row, 1].set_title(f"{name}: Loss")
        axes[row, 1].set_xlabel("Epoch")
        axes[row, 1].legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

plot_generic_histories(torch_histories, "09_transformer_train_validation_curves.png")

# TabNet exposes training loss and validation metrics from its own history.
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(tabnet_history.get("loss", []), label="Training loss")
axes[0].plot(tabnet_history.get("train_logloss", []), label="Training log loss")
axes[0].plot(tabnet_history.get("val_logloss", []), label="Validation loss")
axes[0].set_title("TabNet: Loss")
axes[0].legend()
axes[1].plot(tabnet_history.get("train_accuracy", []), label="Training accuracy")
axes[1].plot(tabnet_history.get("val_accuracy", []), label="Validation accuracy")
axes[1].set_title("TabNet: Accuracy")
axes[1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "10_tabnet_train_validation_curves.png", dpi=220, bbox_inches="tight")
plt.show()

## 11. Validation comparison and model freeze

This is the last development-stage comparison. Review validation performance and the predefined proposed FT-Transformer. After running the next cell, do not change features, threshold, hyperparameters, architectures, or checkpoints based on test results.

In [ ]:
train_val_results = pd.DataFrame(train_val_rows)
train_val_results = train_val_results[["Model", "Family", "Split"] + METRIC_COLUMNS]
train_val_results.to_csv(OUTPUT_DIR / "training_validation_results.csv", index=False)
display(
    train_val_results[train_val_results["Split"] == "Validation"]
    .sort_values(["ROC-AUC", "MCC"], ascending=False)
    .round(4)
)

MODEL_REGISTRY_FROZEN = True
FROZEN_MODEL_NAMES = list(model_registry.keys())
print("MODEL REGISTRY FROZEN:", FROZEN_MODEL_NAMES)

# FINAL TEST GATE

The following cells are the first cells that use `y_test` for performance evaluation. Run them only after the model registry is frozen. Test results are descriptive final results, not inputs to another tuning round.

In [ ]:
assert MODEL_REGISTRY_FROZEN
TEST_RESULTS_UNLOCKED = True

test_probabilities = OrderedDict()

# Scikit-learn / boosting models.
for name in list(classical_models) + list(ensemble_models):
    test_probabilities[name] = positive_probability(model_registry[name], X_test_model)

# Keras models.
test_probabilities["MLP"] = keras_models["MLP"].predict(X_test_np, batch_size=4096, verbose=0).reshape(-1)
test_probabilities["1D CNN"] = keras_models["1D CNN"].predict(
    X_test_np[..., None], batch_size=4096, verbose=0
).reshape(-1)

# Custom PyTorch transformers.
test_probabilities["TabTransformer"] = predict_torch(tab_transformer, test_feature_loader)
test_probabilities["FT-Transformer"] = predict_torch(ft_transformer, test_feature_loader)

# TabNet.
test_probabilities["TabNet"] = tabnet.predict_proba(X_test_np)[:, 1]

test_rows = []
for name, probs in test_probabilities.items():
    row = {"Model": name, "Family": model_families[name], "Split": "Testing"}
    row.update(requested_metrics(y_test, probs))
    test_rows.append(row)
final_test_results = pd.DataFrame(test_rows)[["Model", "Family", "Split"] + METRIC_COLUMNS]
final_test_results = final_test_results.sort_values(["ROC-AUC", "MCC"], ascending=False)
final_test_results.to_csv(OUTPUT_DIR / "final_test_results.csv", index=False)
display(final_test_results.round(4))

all_results = pd.concat([train_val_results, final_test_results], ignore_index=True)
all_results.to_csv(OUTPUT_DIR / "all_model_results.csv", index=False)

predictions = pd.DataFrame({"Actual": y_test.to_numpy()}, index=X_test.index)
for name, probs in test_probabilities.items():
    predictions[f"{name} Probability"] = probs
predictions.to_csv(OUTPUT_DIR / "test_predictions.csv", index_label="OriginalRowIndex")

## 12. Final test confusion matrices

In [ ]:
plot_confusion_grid(
    test_probabilities, y_test,
    "Final Test Confusion Matrices - All Models",
    "11_final_test_confusion_matrices_all_models.png"
)

## 13. Combined model ROC curves

In [ ]:
def plot_combined_roc(names, title, filename):
    plt.figure(figsize=(9, 7))
    for name in names:
        probs = test_probabilities[name]
        fpr, tpr, _ = roc_curve(y_test, probs)
        auc_value = roc_auc_score(y_test, probs)
        plt.plot(fpr, tpr, lw=2, label=f"{name} (AUC={auc_value:.3f})")
    plt.plot([0, 1], [0, 1], "--", color="gray", label="Chance")
    plt.xlabel("False Positive Rate")
    plt.ylabel("True Positive Rate / Sensitivity")
    plt.title(title, weight="bold")
    plt.legend(loc="lower right", fontsize=8)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename, dpi=220, bbox_inches="tight")
    plt.show()

plot_combined_roc(
    list(classical_models) + list(ensemble_models),
    "Combined ROC - Machine Learning and Ensemble Models",
    "12_combined_roc_ml_ensemble.png"
)
plot_combined_roc(
    ["MLP", "1D CNN", "TabTransformer", "FT-Transformer", "TabNet"],
    "Combined ROC - Deep Learning and Tabular Transformers",
    "13_combined_roc_deep_transformers.png"
)

## 14. Training/validation curves with one-time final test references

The dashed test lines below are calculated once from the frozen final models. They are not epoch-wise test monitoring.

In [ ]:
curve_histories = OrderedDict()
curve_histories.update(keras_histories)
curve_histories.update(torch_histories)
curve_histories["TabNet"] = tabnet_curve_history

fig, axes = plt.subplots(len(curve_histories), 2, figsize=(13, 4 * len(curve_histories)))
axes = np.atleast_2d(axes)
for row, (name, hist) in enumerate(curve_histories.items()):
    test_metrics = requested_metrics(y_test, test_probabilities[name])
    axes[row, 0].plot(hist["accuracy"], label="Training")
    axes[row, 0].plot(hist["val_accuracy"], label="Validation")
    axes[row, 0].axhline(test_metrics["Accuracy"], color="#D77A47", linestyle="--", label="Final test (once)")
    axes[row, 0].set_title(f"{name}: Accuracy")
    axes[row, 0].set_xlabel("Epoch")
    axes[row, 0].legend()
    axes[row, 1].plot(hist["loss"], label="Training")
    axes[row, 1].plot(hist["val_loss"], label="Validation")
    axes[row, 1].axhline(test_metrics["Log Loss"], color="#D77A47", linestyle="--", label="Final test (once)")
    axes[row, 1].set_title(f"{name}: Loss")
    axes[row, 1].set_xlabel("Epoch")
    axes[row, 1].legend()
plt.tight_layout()
plt.savefig(FIG_DIR / "14_train_val_test_reference_curves.png", dpi=220, bbox_inches="tight")
plt.show()

# 15. Explainability of the proposed FT-Transformer

The explanation background comes from training data only. Explanations are generated after model freezing. SHAP and LIME operate on raw selected-feature values through a prediction wrapper that applies the frozen training-only category maps and scaler.

In [ ]:
def ft_predict_from_raw_numpy(array):
    frame = pd.DataFrame(np.asarray(array), columns=SELECTED_FEATURES)
    x_cat, x_num = make_transformer_arrays(frame)
    temp_loader = make_loader(x_cat, x_num, np.zeros(len(frame)), shuffle=False, batch_size=1024)
    return predict_torch(ft_transformer, temp_loader)

background = X_train_selected.sample(n=min(60, len(X_train_selected)), random_state=RANDOM_STATE)
explain_cases = X_test_selected.sample(n=min(80 if not FAST_MODE else 20, len(X_test_selected)), random_state=RANDOM_STATE)

masker = shap.maskers.Independent(background.to_numpy(), max_samples=len(background))
ft_shap_explainer = shap.Explainer(
    ft_predict_from_raw_numpy,
    masker,
    algorithm="permutation",
    feature_names=SELECTED_FEATURES,
)
ft_shap_values = ft_shap_explainer(
    explain_cases.to_numpy(),
    max_evals=2 * len(SELECTED_FEATURES) + 1,
)

shap.plots.bar(ft_shap_values, max_display=len(SELECTED_FEATURES), show=False)
plt.title("FT-Transformer SHAP Global Importance")
plt.tight_layout()
plt.savefig(FIG_DIR / "15_ft_transformer_shap_bar.png", dpi=220, bbox_inches="tight")
plt.show()

shap.plots.beeswarm(ft_shap_values, max_display=len(SELECTED_FEATURES), show=False)
plt.title("FT-Transformer SHAP Beeswarm")
plt.tight_layout()
plt.savefig(FIG_DIR / "16_ft_transformer_shap_beeswarm.png", dpi=220, bbox_inches="tight")
plt.show()

## 16. LIME individual explanation

In [ ]:
lime_train = X_train_selected.sample(n=min(10_000, len(X_train_selected)), random_state=RANDOM_STATE)
categorical_indices_lime = [SELECTED_FEATURES.index(c) for c in selected_categorical]
categorical_names_lime = {}
for idx, c in zip(categorical_indices_lime, selected_categorical):
    # Category labels are derived only from the training partition.
    max_code = int(X_train_selected[c].max())
    categorical_names_lime[idx] = [str(v) for v in range(max_code + 1)]

lime_explainer = LimeTabularExplainer(
    training_data=lime_train.to_numpy(dtype=float),
    feature_names=SELECTED_FEATURES,
    class_names=["No diabetes", "Prediabetes/diabetes"],
    categorical_features=categorical_indices_lime,
    categorical_names=categorical_names_lime,
    mode="classification",
    discretize_continuous=True,
    random_state=RANDOM_STATE,
)

def lime_predict_proba(array):
    p1 = ft_predict_from_raw_numpy(array)
    return np.column_stack([1 - p1, p1])

case_position = 0
lime_case = X_test_selected.iloc[case_position].to_numpy(dtype=float)
lime_exp = lime_explainer.explain_instance(
    lime_case, lime_predict_proba, num_features=len(SELECTED_FEATURES)
)
fig = lime_exp.as_pyplot_figure()
fig.suptitle("FT-Transformer LIME Explanation - One Test Case", y=1.02, weight="bold")
fig.tight_layout()
fig.savefig(FIG_DIR / "17_ft_transformer_lime_case.png", dpi=220, bbox_inches="tight")
plt.show()
display(pd.DataFrame(lime_exp.as_list(), columns=["Condition", "LIME weight"]))

## 17. FT-Transformer attention-weight heatmaps

In [ ]:
ft_transformer.eval()
n_attention_cases = min(30, len(X_test_selected))
case_cat = torch.as_tensor(X_test_cat[:n_attention_cases], dtype=torch.long, device=DEVICE)
case_num = torch.as_tensor(X_test_num[:n_attention_cases], dtype=torch.float32, device=DEVICE)
with torch.no_grad():
    _, attention_layers = ft_transformer(case_cat, case_num, return_attention=True)

# Shape after stacking: layers x cases x heads x tokens x tokens.
attn = torch.stack(attention_layers).cpu().numpy()
mean_layer_head = attn.mean(axis=(0, 2))  # cases x tokens x tokens
token_names = ["[CLS]"] + TRANSFORMER_CAT + TRANSFORMER_NUM

# Individual [CLS]-to-feature attention.
cls_to_feature = mean_layer_head[:, 0, 1:]
plt.figure(figsize=(max(10, len(token_names) * 0.65), 7))
sns.heatmap(cls_to_feature, cmap="YlGnBu", xticklabels=token_names[1:], yticklabels=[f"Case {i}" for i in range(n_attention_cases)])
plt.title("FT-Transformer [CLS]-to-Feature Attention by Test Case")
plt.xlabel("Feature token")
plt.ylabel("Untouched test case")
plt.tight_layout()
plt.savefig(FIG_DIR / "18_ft_transformer_cls_attention_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

# Global feature-to-feature attention excluding [CLS].
global_feature_attention = mean_layer_head[:, 1:, 1:].mean(axis=0)
plt.figure(figsize=(10, 8))
sns.heatmap(global_feature_attention, cmap="mako", xticklabels=token_names[1:], yticklabels=token_names[1:])
plt.title("FT-Transformer Mean Feature-to-Feature Attention")
plt.tight_layout()
plt.savefig(FIG_DIR / "19_ft_transformer_global_attention_heatmap.png", dpi=220, bbox_inches="tight")
plt.show()

## 18. Export results and reproducibility information

In [ ]:
version_lines = [
    f"python={sys.version}",
    f"platform={platform.platform()}",
    f"pandas={pd.__version__}",
    f"numpy={np.__version__}",
    f"scikit-learn={__import__('sklearn').__version__}",
    f"imbalanced-learn={__import__('imblearn').__version__}",
    f"xgboost={__import__('xgboost').__version__}",
    f"lightgbm={__import__('lightgbm').__version__}",
    f"catboost={__import__('catboost').__version__}",
    f"tensorflow={tf.__version__}",
    f"torch={torch.__version__}",
    f"shap={shap.__version__}",
    f"dataset_sha256={sha256}",
    f"random_state={RANDOM_STATE}",
    f"fast_mode={FAST_MODE}",
    f"drop_exact_duplicates={DROP_EXACT_DUPLICATES}",
]
(OUTPUT_DIR / "environment_versions.txt").write_text("\n".join(version_lines))

archive_path = shutil.make_archive("/content/diabetes_xai_outputs", "zip", OUTPUT_DIR)
print("Saved:", archive_path)

try:
    from google.colab import files
    files.download(archive_path)
except Exception:
    pass

## 19. Reporting checklist

- Report whether `FAST_MODE` and `DROP_EXACT_DUPLICATES` were enabled.
- Report the exact selected-feature list and random seed.
- Report all nine requested metrics together; Recall and Sensitivity are intentionally identical.
- Do not tune any model after viewing the final test table.
- Interpret SHAP, LIME, and attention as explanations of model behavior, not as causal medical effects.
- This research model is for screening-method comparison, not clinical diagnosis.